# 🐔 PoultrySense - Master Colab Model Trainer (YOLO26n)

This notebook trains two **YOLO26n** models for **PoultrySense-Android-Studio** using the ChickenVerse dataset in your Google Drive (`chicken_project/`):
1. **Chicken Detection Model (`PoultrySense_ChickenDet_v4_26n.pt`)**: Uses the **YOLO26n** architecture (`yolo26n.pt`) with genuine NMS-free end2end detection head, fine-tuned on **ChickenDet** (5,222 images with 141,000+ chicken bounding boxes) with high resolution (`imgsz=960`), SGD optimizer, early stopping, and motion augmentations (`degrees`, `translate`, `scale`, `shear`, `mosaic`).
2. **Chicken Behavior Model (`PoultrySense_BehaviorModel_v2_26n.pt`)**: Uses the **YOLO26n-cls** architecture (`yolo26n-cls.pt`) to classify chicken behaviors (`peck`, `stand`, `walk`) from the 15,000 action clips in **ChickenAct**.

---

## Step 1: Install Dependencies & Check GPU

In [ ]:
# Install Ultralytics supporting YOLO26 (v8.4.0+)
%pip install -q "ultralytics>=8.4.0" opencv-python matplotlib pyyaml tqdm

import sys, os, torch, shutil, glob
from ultralytics import YOLO

print('Python     :', sys.version.split()[0])
print('PyTorch    :', torch.__version__)
print('CUDA avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device :', torch.cuda.get_device_name(0))
else:
    print('WARNING: Running on CPU! Enable GPU in Colab: Runtime -> Change runtime type -> T4 GPU')

## Step 2: Mount Google Drive & Locate Datasets in `chicken_project`
Locates `ChickenDet.zip` (3.7 GB) and `ChickenAct.zip` (1.84 GB).

In [ ]:
import os, glob

# Check if Google Drive is already mounted
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print('drive.mount() error:', e)
        print('👉 Tip: Mount directly by clicking the Folder icon on the left sidebar in Colab -> Drive logo!')

assert os.path.exists('/content/drive/MyDrive'), (
    'Google Drive is not mounted! Click the Folder icon on the left sidebar in Colab, '
    'then click the Mount Drive button and click Connect to Google Drive.'
)

PRIMARY_DIR = '/content/drive/MyDrive/chicken_project'
SEARCH_DIRS = [
    PRIMARY_DIR,
    '/content/drive/MyDrive',
    '/content/drive/MyDrive/PoultrySense',
]

def find_file(filename):
    for d in SEARCH_DIRS:
        direct = os.path.join(d, filename)
        if os.path.exists(direct):
            return direct
        if os.path.exists(d):
            for f in os.listdir(d):
                if f.lower() == filename.lower():
                    return os.path.join(d, f)
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        for f in files:
            if f.lower() == filename.lower():
                return os.path.join(root, f)
    return None

det_zip = find_file('ChickenDet.zip')
act_zip = find_file('ChickenAct.zip')

print('Detection ZIP :', det_zip)
if det_zip:
    print('  Size:', round(os.path.getsize(det_zip) / (1024**3), 2), 'GB')

print('Behavior ZIP  :', act_zip)
if act_zip:
    print('  Size:', round(os.path.getsize(act_zip) / (1024**3), 2), 'GB')

assert det_zip is not None, 'Could not find ChickenDet.zip in Google Drive!'
assert act_zip is not None, 'Could not find ChickenAct.zip in Google Drive!'

print('\n✅ Both ZIP files found in Google Drive!')


## Step 3: Fast Extraction & Automatic COCO-to-YOLO Label Generation

ChickenDet is packaged in COCO JSON format (`annotations/instances_*.json`).
This step extracts the files and converts all 141,000+ COCO bounding boxes into YOLO `.txt` labels (`labels/Train`, `labels/Validation`, `labels/Test`).

In [ ]:
import zipfile, json, yaml

DATASET_ROOT = '/content/datasets'
DET_DIR = os.path.join(DATASET_ROOT, 'ChickenDet')
ACT_DIR = os.path.join(DATASET_ROOT, 'ChickenAct')

# 1. Extract ChickenDet.zip
if not os.path.exists(os.path.join(DET_DIR, 'images', 'Train')):
    print(f'Extracting {os.path.basename(det_zip)} into {DET_DIR} (~20-40 seconds)...')
    os.makedirs(DET_DIR, exist_ok=True)
    with zipfile.ZipFile(det_zip, 'r') as z:
        z.extractall(DET_DIR)
    print('  ChickenDet unzipped successfully!')
else:
    print('ChickenDet images already unzipped.')

# 2. Convert COCO JSON annotations to YOLO format labels
def convert_coco_to_yolo(json_path, split_name):
    if not os.path.exists(json_path):
        print(f'Notice: {json_path} not found.')
        return 0, 0
    
    labels_dir = os.path.join(DET_DIR, 'labels', split_name)
    os.makedirs(labels_dir, exist_ok=True)
    
    with open(json_path, 'r') as f:
        coco = json.load(f)
    
    images = {img['id']: img for img in coco.get('images', [])}
    
    ann_by_img = {}
    for ann in coco.get('annotations', []):
        img_id = ann['image_id']
        ann_by_img.setdefault(img_id, []).append(ann)
    
    converted_anns = 0
    for img_id, img_info in images.items():
        w = img_info['width']
        h = img_info['height']
        fname = img_info['file_name']
        stem = os.path.splitext(fname)[0]
        out_txt = os.path.join(labels_dir, f'{stem}.txt')
        
        lines = []
        for ann in ann_by_img.get(img_id, []):
            x, y, bw, bh = ann['bbox']
            xc = max(0.0, min(1.0, (x + bw / 2.0) / w))
            yc = max(0.0, min(1.0, (y + bh / 2.0) / h))
            nw = max(0.0, min(1.0, bw / w))
            nh = max(0.0, min(1.0, bh / h))
            lines.append(f'0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}')
            converted_anns += 1
            
        with open(out_txt, 'w') as out_f:
            out_f.write('\n'.join(lines) + ('\n' if lines else ''))
            
    print(f'  Converted {split_name}: {len(images)} images, {converted_anns} chicken bounding boxes -> {labels_dir}')
    return len(images), converted_anns

print('\nConverting COCO JSON annotations to YOLO format...')
convert_coco_to_yolo(os.path.join(DET_DIR, 'annotations', 'instances_Train.json'), 'Train')
convert_coco_to_yolo(os.path.join(DET_DIR, 'annotations', 'instances_Validation.json'), 'Validation')
convert_coco_to_yolo(os.path.join(DET_DIR, 'annotations', 'instances_Test.json'), 'Test')

# 3. Write data.yaml pointing to the dataset
DET_YAML = os.path.join(DET_DIR, 'data.yaml')
data_config = {
    'path': DET_DIR,
    'train': 'images/Train',
    'val': 'images/Validation',
    'test': 'images/Test',
    'nc': 1,
    'names': ['chicken']
}
with open(DET_YAML, 'w') as f:
    yaml.safe_dump(data_config, f)

print(f'\nGenerated verified data.yaml:\n{open(DET_YAML).read()}')

## Step 4: Train Chicken Detection Model with YOLO26n (`PoultrySense_ChickenDet_v4_26n.pt`)

**Why YOLO26n:**
- 100% compatible with PoultrySense's current v3 architecture.
- Genuine NMS-free end2end/one2one head (eliminates post-processing latency).
- High resolution `imgsz=960` retains sharp boundaries for fast-moving chickens.
- `epochs=40`, `patience=15` with SGD optimizer on 141,000+ bounding boxes.

In [ ]:
# Path to checkpoints on Google Drive and local runs
RUNS_DIR = '/content/drive/MyDrive/chicken_project/runs'
DET_CHECKPOINT = os.path.join(RUNS_DIR, 'det_chicken_v4_26n', 'weights', 'last.pt')
LOCAL_CHECKPOINT = '/content/runs/det_chicken_v4_26n/weights/last.pt'
os.makedirs(RUNS_DIR, exist_ok=True)

# Check if already completed
completed_weights = [
    os.path.join(RUNS_DIR, 'det_chicken_v4_26n', 'weights', 'best.pt'),
    '/content/runs/det_chicken_v4_26n/weights/best.pt',
    '/content/drive/MyDrive/chicken_project/models/PoultrySense_ChickenDet_v4_26n.pt'
]
already_trained = any(os.path.exists(w) for w in completed_weights)

if already_trained:
    BEST_DET_WEIGHTS = next(w for w in completed_weights if os.path.exists(w))
    print(f'🎉 Training ALREADY COMPLETE! Loaded: {BEST_DET_WEIGHTS}')
elif os.path.exists(DET_CHECKPOINT) or os.path.exists(LOCAL_CHECKPOINT):
    ckpt = DET_CHECKPOINT if os.path.exists(DET_CHECKPOINT) else LOCAL_CHECKPOINT
    print(f'🔄 Resuming training from checkpoint: {ckpt}')
    det_model = YOLO(ckpt)
    det_results = det_model.train(resume=True)
    BEST_DET_WEIGHTS = ckpt.replace('last.pt', 'best.pt')
else:
    print('🚀 Starting fresh Chicken Detection Training with YOLO26n...')
    det_model = YOLO('yolo26n.pt')
    det_results = det_model.train(
        data=DET_YAML,
        epochs=25,       # 25 epochs is optimal for 5,200 images & finishes fast within T4 quota
        patience=10,
        imgsz=960,
        batch=16,
        optimizer='SGD',
        lr0=0.01,
        cache=False,
        
        # Motion & posture augmentations
        degrees=15.0,
        translate=0.2,
        scale=0.5,
        shear=10.0,
        perspective=0.0005,
        mosaic=1.0,
        mixup=0.15,
        
        project=RUNS_DIR,
        name='det_chicken_v4_26n',
        exist_ok=True,
        save_period=1,   # Auto-saves checkpoint to Google Drive after EVERY epoch!
        seed=0
    )
    BEST_DET_WEIGHTS = os.path.join(RUNS_DIR, 'det_chicken_v4_26n', 'weights', 'best.pt')

# Ensure BEST_DET_WEIGHTS is found
for w in completed_weights:
    if os.path.exists(w):
        BEST_DET_WEIGHTS = w
        break

# Immediately backup to Google Drive models folder so it is NEVER lost
DRIVE_MODELS = '/content/drive/MyDrive/chicken_project/models'
os.makedirs(DRIVE_MODELS, exist_ok=True)
if 'BEST_DET_WEIGHTS' in locals() and os.path.exists(BEST_DET_WEIGHTS):
    shutil.copy(BEST_DET_WEIGHTS, os.path.join(DRIVE_MODELS, 'PoultrySense_ChickenDet_v4_26n.pt'))
    shutil.copy(BEST_DET_WEIGHTS, os.path.join(DRIVE_MODELS, 'PoultrySense_ChickenDet_v4.pt'))
    print(f'✅ Safely saved to Google Drive: {DRIVE_MODELS}/PoultrySense_ChickenDet_v4_26n.pt')

print('\nDetection Training Complete!')
print('Best Detection Weights Saved At:', BEST_DET_WEIGHTS)


## Step 5: Evaluate Detection Model Accuracy on Validation Set

In [ ]:
# Locate best.pt across Drive models, Drive runs, and local storage
candidates = [
    getattr(globals(), 'BEST_DET_WEIGHTS', None),
    '/content/drive/MyDrive/chicken_project/models/PoultrySense_ChickenDet_v4_26n.pt',
    '/content/drive/MyDrive/chicken_project/runs/det_chicken_v4_26n/weights/best.pt',
    '/content/runs/det_chicken_v4_26n/weights/best.pt'
]
BEST_DET_WEIGHTS = next((w for w in candidates if w and os.path.exists(w)), None)
assert BEST_DET_WEIGHTS is not None, 'Could not find best.pt weights! Ensure Step 4 completed.'

print('Evaluating weights:', BEST_DET_WEIGHTS)
best_det = YOLO(BEST_DET_WEIGHTS)
m = best_det.val(data=DET_YAML)
print(f'YOLO26n Detection Metrics -> Precision: {m.box.mp:.4f} | Recall: {m.box.mr:.4f} | mAP50: {m.box.map50:.4f} | mAP50-95: {m.box.map:.4f}')


## Step 6: Train Chicken Behavior Classification Model with YOLO26n-cls (`PoultrySense_BehaviorModel_v2_26n.pt`)

Extracts representative frames from `ChickenAct.zip` and maps them to PoultrySense core behaviors (`peck`, `stand`, `walk`), then trains using **YOLO26n-cls** (`yolo26n-cls.pt`).

In [ ]:
import cv2, random

# 1. Extract ChickenAct.zip
if not os.path.exists(ACT_DIR) or len(os.listdir(ACT_DIR)) == 0:
    print(f'Extracting {os.path.basename(act_zip)} into {ACT_DIR} (~15-30 seconds)...')
    os.makedirs(ACT_DIR, exist_ok=True)
    with zipfile.ZipFile(act_zip, 'r') as z:
        z.extractall(ACT_DIR)
    print('  ChickenAct unzipped successfully!')
else:
    print('ChickenAct already unzipped.')

# 2. Extract frames from action video clips into classification dataset
BEHAVIOR_DIR = '/content/datasets/ChickenBehavior_Dataset'
train_dir = os.path.join(BEHAVIOR_DIR, 'train')
val_dir = os.path.join(BEHAVIOR_DIR, 'val')

# Map 15 actions to PoultrySense core behaviors (peck, stand, walk)
ACTION_MAP = {
    'feeding': 'peck',
    'litter_pecking': 'peck',
    'standing': 'stand',
    'resting': 'stand',
    'walking': 'walk',
    'running': 'walk',
}

for split in [train_dir, val_dir]:
    for cls_name in set(ACTION_MAP.values()):
        os.makedirs(os.path.join(split, cls_name), exist_ok=True)

print('\nExtracting representative frames from behavior video clips...')
random.seed(0)
extracted_counts = {'peck': 0, 'stand': 0, 'walk': 0}

for act_folder, target_class in ACTION_MAP.items():
    folder_path = os.path.join(ACT_DIR, act_folder)
    if not os.path.exists(folder_path):
        continue
    
    videos = glob.glob(f'{folder_path}/*.mp4')
    random.shuffle(videos)
    sample_videos = videos[:400]  # Balanced set of 400 clips per behavior for fast training
    
    split_idx = int(len(sample_videos) * 0.8)
    for idx, v_path in enumerate(sample_videos):
        split = 'train' if idx < split_idx else 'val'
        v_stem = os.path.splitext(os.path.basename(v_path))[0]
        out_jpg = os.path.join(BEHAVIOR_DIR, split, target_class, f'{act_folder}_{v_stem}.jpg')
        if os.path.exists(out_jpg):
            continue
            
        cap = cv2.VideoCapture(v_path)
        f_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if f_count > 0:
            cap.set(cv2.CAP_PROP_POS_FRAMES, f_count // 2)
            ret, frame = cap.read()
            if ret and frame is not None:
                cv2.imwrite(out_jpg, frame)
                extracted_counts[target_class] += 1
        cap.release()

print('Extracted behavior image frames:', extracted_counts)

# 3. Train Behavior Classification Model using YOLO26n-cls
ACT_CHECKPOINT = os.path.join(RUNS_DIR, 'behavior_chicken_v2_26n', 'weights', 'last.pt')
if os.path.exists(ACT_CHECKPOINT):
    print(f'🔄 CHECKPOINT FOUND: {ACT_CHECKPOINT}')
    print('Resuming behavior training from where you left off...')
    act_model = YOLO(ACT_CHECKPOINT)
    act_results = act_model.train(resume=True)
else:
    print('🚀 Starting fresh Behavior Training with YOLO26n-cls...')
    act_model = YOLO('yolo26n-cls.pt')
    act_results = act_model.train(
        data=BEHAVIOR_DIR,
        epochs=25,       # 25 epochs runs in ~5-8 minutes
        patience=8,
        imgsz=224,
        batch=32,
        project=RUNS_DIR,
        name='behavior_chicken_v2_26n',
        exist_ok=True,
        save_period=1,
        seed=0
    )

BEST_ACT_WEIGHTS = os.path.join(RUNS_DIR, 'behavior_chicken_v2_26n', 'weights', 'best.pt')
# Immediate backup to Drive models folder
DRIVE_MODELS = '/content/drive/MyDrive/chicken_project/models'
os.makedirs(DRIVE_MODELS, exist_ok=True)
if os.path.exists(BEST_ACT_WEIGHTS):
    shutil.copy(BEST_ACT_WEIGHTS, os.path.join(DRIVE_MODELS, 'PoultrySense_BehaviorModel_v2_26n.pt'))
    shutil.copy(BEST_ACT_WEIGHTS, os.path.join(DRIVE_MODELS, 'PoultrySense_BehaviorModel_v2.pt'))
    print(f'✅ Safely saved behavior model to Google Drive: {DRIVE_MODELS}/PoultrySense_BehaviorModel_v2_26n.pt')

print('\nBehavior Training Complete!')
print('Best Behavior Weights Saved At:', BEST_ACT_WEIGHTS)


## Step 7: Export Trained Models Back to Google Drive (`chicken_project/models/`)

In [ ]:
DRIVE_OUT = '/content/drive/MyDrive/chicken_project/models'
os.makedirs(DRIVE_OUT, exist_ok=True)

def safe_copy(src, dst):
    if src and os.path.exists(src) and os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)

# 1. Save Detection Model
dst_det_26n = os.path.join(DRIVE_OUT, 'PoultrySense_ChickenDet_v4_26n.pt')
dst_det = os.path.join(DRIVE_OUT, 'PoultrySense_ChickenDet_v4.pt')
safe_copy(BEST_DET_WEIGHTS, dst_det_26n)
safe_copy(BEST_DET_WEIGHTS, dst_det)
if os.path.exists(dst_det_26n):
    print(f'✅ Detection Model saved to Drive: {dst_det_26n} ({os.path.getsize(dst_det_26n)/1e6:.1f} MB)')

# 2. Save Behavior Model
dst_act_26n = os.path.join(DRIVE_OUT, 'PoultrySense_BehaviorModel_v2_26n.pt')
dst_act = os.path.join(DRIVE_OUT, 'PoultrySense_BehaviorModel_v2.pt')
safe_copy(BEST_ACT_WEIGHTS, dst_act_26n)
safe_copy(BEST_ACT_WEIGHTS, dst_act)
if os.path.exists(dst_act_26n):
    print(f'✅ Behavior Model saved to Drive: {dst_act_26n} ({os.path.getsize(dst_act_26n)/1e6:.1f} MB)')

print('\n🎉 BOTH TRAINED YOLO26n MODELS SUCCESSFULLY VERIFIED IN YOUR GOOGLE DRIVE (chicken_project/models/)!')


## Step 8: How to use these models in `PoultrySense-Android-Studio`

1. Open your Google Drive: **`My Drive > chicken_project > models/`**
2. Download `PoultrySense_ChickenDet_v4_26n.pt` and `PoultrySense_BehaviorModel_v2_26n.pt`.
3. Place them into your local project folder: `poultrysense_ai/models/`.
4. Restart your Python AI server using `start_ai_server.bat`.